# Phase 9 — Asynchronous Processing, Durable Jobs, Batch Workflows and Performance

## 1. Phase Overview

Phase 9 changed how VIGILOX executed document-processing work.

Earlier versions of the application could process documents synchronously, meaning a browser request could remain connected while OCR, structured extraction, validation, and persistence were being performed.

That model was acceptable during early development, but it was not suitable for a production-oriented document-processing workflow.

OCR is CPU-intensive, LLM requests depend on an external provider, and a complete document pipeline may take many seconds.

Phase 9 therefore introduced a durable asynchronous processing architecture.

The new flow became:

```text
Browser
   ↓
FastAPI
   ↓
Create Durable Job
   ↓
PostgreSQL
   ↓
Return Job ID
   ↓
Background Worker
   ↓
OCR
   ↓
Structured Extraction
   ↓
Validation
   ↓
Persistence
   ↓
Completed Document
```

The browser no longer needed to remain connected while the expensive processing pipeline was running.

---

# 2. Objective

The primary objective of Phase 9 was:

> Move expensive document processing out of the HTTP request lifecycle and into a durable PostgreSQL-backed worker system.

The phase also aimed to improve:

- API responsiveness
- processing reliability
- failure recovery
- provider retry handling
- batch uploads
- queue visibility
- browser polling
- duplicate work prevention
- worker concurrency behavior
- pipeline initialization
- performance measurement

---

# 3. Problems Before Phase 9

The synchronous processing model created several operational problems.

## 3.1 Long HTTP Requests

A document could require:

```text
Image preprocessing
      +
OCR
      +
LLM request
      +
Validation
      +
Database persistence
```

before the HTTP request could finish.

A slow pipeline therefore became a slow API request.

---

## 3.2 Browser Dependency

If the processing lifecycle was coupled to the browser request, a user closing the page or losing connectivity could make the interaction unreliable.

The desired rule was:

> Browser sessions should not be job queues.

---

## 3.3 Limited Retry Control

External LLM requests can fail because of:

- rate limits
- temporary provider failures
- network timeouts
- connection failures

These failures needed to be represented as durable job state rather than hidden inside one long request.

---

## 3.4 No Durable Queue

A process-local background task would not be sufficient.

If the API restarted, in-memory jobs could disappear.

The queue therefore needed to live in the system of record:

```text
PostgreSQL
```

---

## 3.5 Batch Processing Needed Isolation

A batch of documents should not behave as one indivisible request.

If:

```text
Document A succeeds
Document B fails
Document C succeeds
```

then A and C should remain successful.

One bad file should not invalidate the entire batch.

---

# 4. Design Principles

Phase 9 followed several core principles.

## 4.1 Durable Before Convenient

Job state must survive:

```text
Browser close
API restart
Worker restart
Temporary provider failure
```

Therefore PostgreSQL became the durable queue.

---

## 4.2 Small Job State Model

The queue uses a deliberately small set of authoritative states:

```text
QUEUED
PROCESSING
RETRY_WAIT
COMPLETED
FAILED
```

Additional processing detail is represented separately through fields such as:

```text
current_stage
attempt_count
max_attempts
next_attempt_at
error_code
```

This avoids creating dozens of fragile state values.

---

## 4.3 No Fake Progress Percentages

The system does not display artificial values such as:

```text
37% complete
61% complete
84% complete
```

when the backend cannot meaningfully calculate them.

Instead, the user sees real processing state and advisory stage information.

---

## 4.4 Retry Only Retryable Failures

Not every failure deserves another attempt.

For example:

```text
Rate limit
→ Retryable

Provider 5xx
→ Retryable

Temporary connection failure
→ Retryable

Unsupported document
→ Domain outcome

Missing source
→ Not fixed by retrying the provider
```

Retry behavior therefore depends on failure classification.

---

# 5. Durable Job Architecture

The new architecture separated request handling from document execution.

```text
┌────────────────────┐
│      Browser       │
└─────────┬──────────┘
          │
          ▼
┌────────────────────┐
│      FastAPI       │
│                    │
│ Create Job         │
│ Return Job ID      │
└─────────┬──────────┘
          │
          ▼
┌────────────────────┐
│     PostgreSQL     │
│                    │
│ Job Queue          │
│ Attempts           │
│ Lease              │
│ Retry Schedule     │
└─────────┬──────────┘
          │
          ▼
┌────────────────────┐
│       Worker       │
│                    │
│ Claim Job          │
│ Process Document   │
│ Persist Result     │
└────────────────────┘
```

The API became responsible for accepting work.

The worker became responsible for performing work.

---

# 6. Main Async Endpoint

The primary asynchronous document endpoint became:

```http
POST /api/v1/document-jobs
```

The endpoint performs lightweight request work such as:

```text
Validate Upload
      ↓
Persist Pending Source
      ↓
Create Job
      ↓
Return Job Metadata
```

The expensive OCR/LLM pipeline is not executed inside this request.

---

# 7. Compatibility Endpoint

The synchronous endpoint was retained for compatibility:

```http
POST /api/v1/documents/analyze
```

This allowed existing integrations and earlier workflows to continue functioning.

However, the main browser workflow moved to:

```text
POST /api/v1/document-jobs
```

The asynchronous API became the preferred processing path.

---

# 8. Job State Model

The durable processing lifecycle is:

```text
QUEUED
   ↓
PROCESSING
   ├── success ─────────────→ COMPLETED
   │
   ├── retryable failure ───→ RETRY_WAIT
   │                              ↓
   │                            QUEUED
   │                              ↓
   │                          PROCESSING
   │
   └── terminal failure ─────→ FAILED
```

This model keeps retries visible and durable.

---

# 9. Job Status API

The frontend can query job state without needing access to worker internals.

A job-status response exposes safe operational information such as:

```text
status
current_stage
attempt_count
max_attempts
next_attempt_at
error_code
document_id
```

Sensitive provider or internal exception details are not exposed directly to the browser.

---

# 10. Browser Polling

After creating a job, the browser polls the job-status endpoint.

Conceptually:

```text
POST document job
      ↓
Receive job_id
      ↓
Wait
      ↓
GET job status
      ↓
QUEUED / PROCESSING?
      ├── Yes → poll again
      └── No
           ↓
      COMPLETED / FAILED
```

The frontend uses bounded polling rather than aggressive continuous requests.

A polling interval of approximately:

```text
1500 ms
```

was used for the browser workflow.

The goal was to provide responsive feedback without unnecessarily loading the API.

---

# 11. Persistent Upload Preview

Phase 9 also improved the upload experience.

The selected document remained visible while the job moved through states such as:

```text
Ready
Queued
Processing
Completed
```

This prevented the interface from feeling disconnected from the document the user had just submitted.

Batch rows intentionally remained simpler and did not require full document thumbnails.

---

# 12. Pending Upload Storage

Asynchronous processing introduced a new storage problem.

The uploaded bytes need to exist somewhere between:

```text
HTTP Upload
     ↓
Job Creation
     ↓
Worker Claim
```

The file cannot immediately be treated as a managed persisted document because the document record may not exist yet.

Therefore Phase 9 separated:

```text
Pending Storage
       ≠
Managed Document Storage
```

Conceptually:

```text
Upload
   ↓
Pending Storage
   ↓
Worker Processing
   ↓
Successful Persistence
   ↓
Managed Storage
```

This separation prevents storage reconciliation logic from treating in-flight uploads as orphaned managed documents.

---

# 13. PostgreSQL Job Queue

PostgreSQL was used as the job queue rather than introducing a second infrastructure system.

The queue stores durable state including:

- job ID
- status
- attempt count
- maximum attempts
- retry time
- lease information
- processing stage
- source reference
- resulting document ID
- safe failure code

This kept the architecture centered around the existing system of record.

---

# 14. Safe Job Claiming

Workers need to claim jobs without two workers processing the same queued item simultaneously.

PostgreSQL row locking was used:

```sql
FOR UPDATE SKIP LOCKED
```

Conceptually:

```text
Worker A ─┐
          ├─→ PostgreSQL Queue
Worker B ─┘

Worker A locks Job 1
Worker B skips Job 1
Worker B claims Job 2
```

This allows multiple worker processes to compete safely for work.

---

# 15. Worker Lease

A worker claim cannot remain valid forever.

If a worker crashes while holding a job, another worker eventually needs to recover it.

A lease mechanism was therefore introduced.

Conceptually:

```text
Worker claims job
      ↓
Lease starts
      ↓
Worker processes job
      │
      ├── completes → authoritative completion
      │
      └── dies
             ↓
         lease expires
             ↓
         job becomes recoverable
```

The final configured lease duration was:

```text
360 seconds
```

This was selected to remain safely above the measured and bounded pipeline execution budget.

---

# 16. Stale Worker Protection

A stale worker must not overwrite a valid result produced after its lease expired.

Completion therefore remains scoped to the worker that still holds the authoritative claim.

Conceptually:

```text
Worker A claims Job
      ↓
Lease expires
      ↓
Worker B reclaims Job
      ↓
Worker B completes Job
      ↓
Worker A finally returns late
      ↓
Worker A cannot overwrite Worker B
```

This protects the job record against delayed stale completion.

---

# 17. Worker Concurrency

The default worker concurrency was kept at:

```text
1
```

This decision was based on the nature of the workload.

PaddleOCR is:

- CPU-intensive
- internally multi-threaded
- the dominant local processing cost

Increasing worker concurrency on the same CPU can cause two OCR operations to compete for the same cores.

The rule became:

> Increase concurrency only after measuring throughput on the target machine.

Concurrency should not be derived blindly from CPU count.

---

# 18. Retry Architecture

Phase 9 separated two different retry layers.

## Extraction-Level Retry

This handles structured-output recovery inside one job attempt.

For example:

```text
Provider returns malformed structured output
        ↓
Bounded extraction recovery
```

---

## Job-Level Retry

This handles transient operational failures.

For example:

```text
429 Rate Limit
      ↓
RETRY_WAIT
      ↓
Retry later
```

Other examples include:

```text
Provider 5xx
Connection failure
Temporary timeout
Temporary infrastructure failure
```

Separating the layers prevents one retry mechanism from doing all kinds of recovery.

---

# 19. Retry State

Retryable failures move the job into:

```text
RETRY_WAIT
```

The job can store:

```text
attempt_count
next_attempt_at
error_code
```

The worker does not immediately spin on the same failed job.

Instead:

```text
Failure
   ↓
Calculate Backoff
   ↓
RETRY_WAIT
   ↓
next_attempt_at reached
   ↓
Job eligible again
```

---

# 20. Retry-After Handling

Provider rate-limit responses may contain retry guidance.

Where available, the system respects provider retry timing rather than immediately retrying.

Conceptually:

```text
Groq 429
   ↓
Retry-After
   ↓
Schedule job retry
   ↓
RETRY_WAIT
```

This reduces unnecessary provider pressure.

---

# 21. Bounded Retries

Retries cannot continue indefinitely.

A job has:

```text
attempt_count
max_attempts
```

Typical configured maximum:

```text
3 attempts
```

When retryable failures continue beyond the configured limit:

```text
FAILED
```

with a safe terminal error code such as:

```text
ATTEMPTS_EXHAUSTED
```

may be recorded.

---

# 22. Groq SDK Retry Behavior

The provider SDK also performs limited internal retry behavior.

The configured SDK retry count was reduced to:

```text
VIGILOX_GROQ_MAX_RETRIES=1
```

The reason was to avoid uncontrolled retry multiplication between:

```text
SDK retry
×
Extraction retry
×
Job retry
```

Retry layers need explicit boundaries so the worst-case processing time remains understandable.

---

# 23. Provider Timeouts

Provider requests were bounded with explicit timeout configuration.

Representative values:

```text
Connect timeout: 5 seconds
Read timeout:    40 seconds
```

This prevents a single external request from blocking a worker indefinitely.

---

# 24. Batch Processing

Phase 9 introduced durable batch workflows.

A batch request can contain multiple files:

```text
File A
File B
File C
File D
```

The batch becomes:

```text
Batch
 ├── Job A
 ├── Job B
 ├── Job C
 └── Job D
```

Each child job remains independently authoritative.

---

# 25. Batch Isolation

The batch model supports partial success.

For example:

```text
Job A → COMPLETED
Job B → FAILED
Job C → COMPLETED
Job D → DUPLICATE
```

The successful documents remain usable.

The failed item does not invalidate successful siblings.

This is important for real intake workflows.

---

# 26. Batch Limits

A maximum number of files per batch was introduced.

Representative configuration:

```text
VIGILOX_MAX_BATCH_FILES=20
```

The limit protects the queue from one request creating an uncontrolled amount of work.

---

# 27. Batch UI

The browser batch workflow exposes each file independently.

Each row can represent:

```text
Filename
Validation State
Job State
Result
Failure
Completed Document Link
```

The UI avoids representing an entire batch with one fake progress value.

---

# 28. API Responsiveness

One of the key goals of asynchronous processing was reducing request latency.

Measured behavior showed approximately:

```text
POST /document-jobs
median ≈ 16 ms
p95    ≈ 21 ms
```

This endpoint only needs to validate and persist job metadata instead of running OCR and LLM extraction before returning.

---

# 29. Job Status Performance

Polling also remained inexpensive.

Representative measurement:

```text
GET /document-jobs/{id}

median ≈ 3 ms
p95    ≈ 9 ms
```

This made bounded browser polling practical.

---

# 30. Worker Processing Performance

Moving processing to a worker does not make OCR inherently faster.

Measured worker processing remained approximately:

```text
median ≈ 17.4 seconds
```

The earlier synchronous pipeline was approximately:

```text
median ≈ 18.4 seconds
```

Therefore the important Phase 9 improvement was:

> responsiveness and architecture

not:

> claiming that CPU OCR became dramatically faster.

---

# 31. Performance Interpretation

The architecture changed:

```text
Before

Browser Request
    ↓
18+ seconds of pipeline work
    ↓
Response
```

into:

```text
After

Browser Request
    ↓
~milliseconds
    ↓
Job ID

Worker
    ↓
~seconds of pipeline work
    ↓
Result
```

The processing cost still exists, but it no longer blocks the HTTP request lifecycle.

---

# 32. Pipeline Initialization

Phase 9 also investigated pipeline startup behavior.

PaddleOCR initialization has a measurable startup cost.

API and worker processes have different needs.

---

## API Process

The main asynchronous API does not need to execute OCR for each document.

Therefore eager OCR initialization can be disabled:

```env
VIGILOX_API_EAGER_PIPELINE=false
```

This reduces API startup overhead.

---

## Worker Process

The worker performs OCR on every document.

For a worker, eager initialization is useful because:

```text
Worker Start
    ↓
Load OCR
    ↓
Ready
    ↓
Claim Job
```

is preferable to:

```text
Claim Job
    ↓
Start Lease
    ↓
Load OCR
    ↓
Process
```

The second approach wastes active lease time on model initialization.

---

# 33. Async Browser State

The frontend needed to represent real backend job state.

Example:

```text
Selected
   ↓
Queued
   ↓
Processing
   ↓
Completed
```

or:

```text
Selected
   ↓
Queued
   ↓
Processing
   ↓
Retry Wait
   ↓
Processing
   ↓
Completed
```

The UI therefore became a projection of durable backend state rather than a local animation.

---

# 34. Failure Presentation

A failed job is represented as a durable terminal state.

The browser can display:

- safe failure status
- failure code
- attempt count
- next action where appropriate

Internal provider exceptions and sensitive details remain server-side.

---

# 35. Browser Can Close Safely

A major architectural result of Phase 9 was:

```text
Upload
  ↓
Job persisted
  ↓
Browser closes
  ↓
Worker continues
  ↓
Document completes
```

This is a key distinction between durable background processing and frontend-controlled processing.

---

# 36. Duplicate Processing Considerations

Async processing also made duplicate protection more important.

Without protection:

```text
Upload same file twice
      ↓
Job A
Job B
      ↓
OCR twice
LLM twice
```

The advanced duplicate-detection system was completed in Phase 10, but Phase 9 established the durable job infrastructure required to handle duplicates correctly.

---

# 37. PostgreSQL as Queue and System of Record

Using PostgreSQL avoided introducing a separate queue service during this stage.

The architecture remained:

```text
PostgreSQL
├── Documents
├── Analysis
├── Reviews
├── Audit History
├── Jobs
└── Batches
```

Advantages included:

- transactional behavior
- durable state
- existing operational familiarity
- fewer infrastructure components
- easier auditability
- direct visibility into job state

---

# 38. Why Not an In-Memory Queue?

An in-memory queue would be simpler initially, but:

```text
Process restart
      ↓
Queue lost
```

That behavior would contradict the reliability requirements of document-processing work.

Phase 9 therefore treated durability as more important than minimal implementation complexity.

---

# 39. Why Not Make the Browser the Queue?

The browser cannot be trusted to remain:

```text
open
connected
awake
on the same page
```

until OCR and LLM processing finish.

The server must own the work lifecycle.

---

# 40. Operational Visibility

Durable jobs made it possible to inspect processing behavior independently of the user interface.

Operators can reason about:

```text
How many jobs are queued?
How many are processing?
Which jobs are retrying?
How many attempts have occurred?
When is the next retry?
Which job produced which document?
```

This later became important for Phase 11 observability.

---

# 41. Key Configuration

Representative Phase 9 configuration includes:

```env
VIGILOX_WORKER_CONCURRENCY=1

VIGILOX_WORKER_POLL_SECONDS=2
VIGILOX_WORKER_IDLE_POLL_SECONDS=5

VIGILOX_JOB_LEASE_SECONDS=360
VIGILOX_JOB_MAX_ATTEMPTS=3

VIGILOX_MAX_BATCH_FILES=20

VIGILOX_API_EAGER_PIPELINE=false

VIGILOX_GROQ_MAX_RETRIES=1
VIGILOX_GROQ_READ_TIMEOUT_SECONDS=40
VIGILOX_GROQ_CONNECT_TIMEOUT_SECONDS=5

VIGILOX_EXTRACTION_ATTEMPTS=3
```

These values were chosen to keep execution bounded and operational behavior understandable.

---

# 42. Important Implementation Components

Phase 9 involved components responsible for:

```text
Job API
Batch API
Job Repository
Worker
Pending Storage
Retry Classification
Provider Backoff
Job Polling
Batch UI
Pipeline Initialization
```

The implementation was spread across the backend service layer, database repositories, worker process, and frontend job-handling modules.

---

# 43. Testing Strategy

Phase 9 required tests beyond simple endpoint correctness.

Important test categories included:

- job creation
- job status transitions
- worker claiming
- concurrent worker claims
- lease behavior
- retry scheduling
- attempt exhaustion
- pending file handling
- batch partial success
- polling contracts
- worker recovery
- stale worker behavior
- structured extraction retry boundaries

The goal was to test the processing lifecycle rather than only individual functions.

---

# 44. Concurrency Testing

Concurrency is one of the highest-risk areas in a worker system.

Important questions included:

```text
Can two workers claim one job?
Can a stale worker overwrite a result?
Can a job be recovered after a dead worker?
Can retries create duplicate work?
Can batch children remain independent?
```

PostgreSQL locking, worker ownership, and lease checks were used to maintain correctness.

---

# 45. Performance Testing

Performance measurement focused on separating:

```text
API latency
```

from:

```text
actual document processing duration
```

This distinction is important.

A fast asynchronous API does not mean the OCR pipeline itself is fast.

Phase 9 therefore avoided misleading performance claims.

---

# 46. What Phase 9 Did Not Try to Solve

Phase 9 did not attempt to:

- replace PaddleOCR with a faster OCR system
- invent fake progress percentages
- automatically scale workers without measurement
- treat every provider error as retryable
- introduce unnecessary distributed infrastructure
- hide failed jobs
- make browser state authoritative

The focus remained durability, responsiveness, and controlled execution.

---

# 47. Lessons Learned

## 47.1 Asynchronous Architecture Improves Responsiveness, Not OCR Speed

Moving OCR into a worker makes the API responsive.

It does not remove the OCR cost.

---

## 47.2 Durable State Matters More Than Background Threads

A background task is not automatically a durable job.

The difference is whether state survives process failure.

---

## 47.3 Retries Need Layers

Provider retries, extraction retries, and job retries solve different problems.

Combining them without limits can produce unpredictable execution times.

---

## 47.4 Concurrency Must Be Measured

More worker processes or threads do not automatically mean more throughput.

For CPU-heavy OCR, uncontrolled concurrency can reduce overall efficiency.

---

## 47.5 Worker Ownership Must Be Explicit

A distributed worker architecture needs to know:

```text
Who owns this job?
Is the lease still valid?
Can this worker still commit the result?
```

Without these rules, stale workers can corrupt state.

---

## 47.6 Batch Success Must Be Per File

Document-processing batches are not transactions where every file must succeed.

Each document needs its own processing outcome.

---

## 47.7 PostgreSQL Can Be a Practical Durable Queue

For this system, PostgreSQL provided enough durability and concurrency control without immediately requiring another infrastructure platform.

---

# 48. Phase 9 Deliverables

Phase 9 produced:

- durable PostgreSQL-backed job queue
- asynchronous document-processing API
- dedicated worker process
- safe worker job claiming
- job leases
- retry scheduling
- bounded attempts
- provider timeout handling
- extraction retry separation
- job-status polling
- pending upload storage
- batch processing
- partial batch success
- browser async states
- persistent upload context
- configurable worker concurrency
- API/worker pipeline initialization controls
- performance measurement
- synchronous compatibility endpoint preservation

---

# 49. Phase 9 Architecture After Completion

```text
                   ┌─────────────────┐
                   │     Browser     │
                   └────────┬────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │     FastAPI     │
                   │                 │
                   │ Upload          │
                   │ Create Job      │
                   │ Job Status      │
                   │ Batch API       │
                   └────────┬────────┘
                            │
                            ▼
              ┌────────────────────────────┐
              │         PostgreSQL         │
              │                            │
              │ Documents                  │
              │ Jobs                       │
              │ Batches                    │
              │ Retry State                │
              │ Lease Ownership            │
              └─────────────┬──────────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │      Worker     │
                   │                 │
                   │ Claim Job       │
                   │ PaddleOCR       │
                   │ Groq            │
                   │ Validation      │
                   │ Persistence     │
                   └────────┬────────┘
                            │
                            ▼
                   ┌─────────────────┐
                   │ Final Document  │
                   └─────────────────┘
```

---

# 50. Final Outcome

Phase 9 converted VIGILOX from a system where expensive document processing could be tied to an HTTP request into a durable asynchronous processing platform.

The major architectural change was:

```text
Before

HTTP Request
    ↓
OCR
    ↓
LLM
    ↓
Validation
    ↓
Persistence
    ↓
HTTP Response
```

becoming:

```text
After

HTTP Request
    ↓
Durable Job
    ↓
Immediate Response

      +

Background Worker
    ↓
OCR
    ↓
LLM
    ↓
Validation
    ↓
Persistence
```

The result was a system with:

- faster API responses
- durable work
- controlled retries
- recoverable worker execution
- batch isolation
- measurable processing behavior
- cleaner separation between web traffic and document intelligence

This architecture became the foundation for the advanced intelligence and duplicate-protection work implemented in the next phase.

```text
Phase 9
Async Processing
Durable Jobs
Performance
Batch Processing
        ↓
Phase 10
Advanced Document Intelligence
Duplicate Detection
Unsupported Documents
Quality Calibration
Extraction Resilience
```

---

## Phase 9 Summary

| Area | Result |
|---|---|
| Durable Job Queue | Implemented |
| PostgreSQL Job Persistence | Implemented |
| Background Worker | Implemented |
| Async Document API | Implemented |
| Job Status API | Implemented |
| Browser Polling | Implemented |
| Worker Leasing | Implemented |
| Safe Job Claims | Implemented |
| Retry Scheduling | Implemented |
| Bounded Attempts | Implemented |
| Pending Upload Storage | Implemented |
| Batch Processing | Implemented |
| Partial Batch Success | Implemented |
| API Lazy OCR Option | Implemented |
| Worker Eager OCR | Implemented |
| Performance Measurement | Completed |
| Sync Compatibility Endpoint | Preserved |

---

**Next:** `Phase 10 — Advanced Document Intelligence`